In [10]:
import pandas as pd
import numpy as np
import time
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [11]:
import importlib
import feature_engineer
importlib.reload(feature_engineer)

<module 'feature_engineer' from '/Users/jiashenwang/Desktop/Lucas_Systems_Capstone_Project/Model_Jiashen/feature_engineer.py'>

In [12]:
# Configurations
warehouses = ["OE", "OF", "RT"]
max_time = 300
models = []
results = []

In [13]:
from feature_engineer import get_engineered_df_allWC

for wh in warehouses:
    DATA_PATH = f"../data/processed/{wh.lower()}_detailed.parquet"
    print(f"--- Training Model for {wh} ---")
        
    # 1. Preprocess
    df, features, cat_cols = get_engineered_df_allWC(DATA_PATH, warehouse=wh, max_time=max_time)
    # remove distance from features
    # features = [f for f in features if f != "Travel_Distance"]
    
    # 2. Prepare Data (One-Hot Encoding)
    X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
    y = df["Time_Delta_sec"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 3. Model Training & Timing
    start_time = time.time()
    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    end_time = time.time()
    models.append(model)
    
    # 4. Evaluation
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    runtime = end_time - start_time
    
    # 5. Store Results
    results.append({
        "Warehouse": wh,
        "Train_Rows": len(X_train),
        "R^2": round(r2, 4),
        "MAE": round(mae, 2),
        "Runtime_Sec": round(runtime, 2)
        })

# Display Clean Results Table
results_df = pd.DataFrame(results)
display(results_df)

--- Training Model for OE ---
--- Training Model for OF ---
--- Training Model for RT ---


,Warehouse,Train_Rows,R^2,MAE,Runtime_Sec
0,OE,72571,0.4986,21.62,1.23
1,OF,96217,0.5075,24.73,1.26
2,RT,64542,0.3559,19.22,1.10


In [ ]:
models_nodist = []
results_nodist = []

for wh in warehouses:
    DATA_PATH = f"../data/processed/{wh.lower()}_detailed.parquet"
    print(f"--- Training Model for {wh} ---")
    
    # 1. Preprocess
    df, features, cat_cols = get_engineered_df_allWC(DATA_PATH, warehouse=wh, max_time=max_time)
    # remove distance, "same_aisle", "same_lockey", "diff_level" from features
    features = [f for f in features if f not in ["Travel_Distance", "same_aisle", "same_lockey", "same_level"]]
    cat_cols = [col for col in cat_cols if col not in ["same_aisle", "same_lockey", "same_level"]]
    
    # 2. Prepare Data (One-Hot Encoding)
    X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
    y = df["Time_Delta_sec"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 3. Model Training & Timing
    start_time = time.time()
    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    end_time = time.time()
    models_nodist.append(model)
    
    # 4. Evaluation
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    runtime = end_time - start_time
    
    # 5. Store Results
    results_nodist.append({
        "Warehouse": wh,
        "Train_Rows": len(X_train),
        "R^2": round(r2, 4),
        "MAE": round(mae, 2),
        "Runtime_Sec": round(runtime, 2)
        })

# Display Clean Results Table
results_df_nodist = pd.DataFrame(results_nodist)
display(results_df_nodist)

--- Training Model for OE ---
--- Training Model for OF ---
--- Training Model for RT ---


,Warehouse,Train_Rows,R^2,MAE,Runtime_Sec
0,OE,72571,0.1385,31.11,1.06
1,OF,96217,0.1259,36.92,1.12
2,RT,64542,0.1596,23.67,0.93


In [23]:
# Append MAE column of nodist to original results_df, right after MAE column and befoire Runtime_Sec, also add percentage increase
display_df = results_df.copy()
display_df["MAE(noDist)"] = results_df_nodist["MAE"]
#display_df["MAE_Increase(%)"] = ((display_df["MAE_noDist"] - display_df["MAE"]) / display_df["MAE"] * 100).round(2)

# Add a mean time column for each warehouse
mean_times = []
for wh in warehouses:
    data_path = f"../data/processed/{wh.lower()}_detailed.parquet"
    d = pd.read_parquet(data_path)
    mean_time = d["Time_Delta_sec"].mean()
    mean_time = round(mean_time, 1)
    mean_times.append(mean_time)
display_df["Mean_Time"] = mean_times

# move Runtime_Sec column to the end
cols = list(display_df.columns)
cols.append(cols.pop(cols.index("Runtime_Sec")))
# do not display r^2
cols.remove("R^2")
display_df = display_df[cols]
display(display_df) 

,Warehouse,Train_Rows,MAE,MAE(noDist),Mean_Time,Runtime_Sec
0,OE,72571,21.62,31.11,72.7,1.23
1,OF,96217,24.73,36.92,81.5,1.26
2,RT,64542,19.22,23.67,50.4,1.10
